In [4]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

In [19]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 10

train_data = tf.keras.utils.image_dataset_from_directory(
    "dataset/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

test_data = tf.keras.utils.image_dataset_from_directory(
    "dataset/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)
train_data = train_data.map(lambda x, y: (preprocess_input(x), y))
test_data = test_data.map(lambda x, y: (preprocess_input(x), y))

Found 1440 files belonging to 6 classes.
Found 360 files belonging to 6 classes.


In [20]:
for x, y in train_data.take(1):
    print(x.shape)   # (16, 224, 224, 3)
    print(y.shape)   # (16, 6)

(16, 224, 224, 3)
(16, 6)


In [21]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(6, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,850,758 (90.98 MB)

 Trainable params: 263,046 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [22]:
early_stop = EarlyStopping(
    patience=3,
    restore_best_weights=True
)

In [23]:
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=EPOCHS,
    callbacks=[early_stop]
)

Epoch 1/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 102s 1s/step - accuracy: 0.9160 - loss: 0.2773 - val_accuracy: 0.9861 - val_loss: 0.0625
Epoch 2/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 97s 1s/step - accuracy: 0.9951 - loss: 0.0218 - val_accuracy: 0.9944 - val_loss: 0.0349
Epoch 3/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 133s 979ms/step - accuracy: 0.9937 - loss: 0.0197 - val_accuracy: 0.9722 - val_loss: 0.0695
Epoch 4/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 93s 1s/step - accuracy: 0.9965 - loss: 0.0124 - val_accuracy: 0.9917 - val_loss: 0.0159
Epoch 5/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 99s 1s/step - accuracy: 0.9986 - loss: 0.0063 - val_accuracy: 0.9833 - val_loss: 0.0463
Epoch 6/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 93s 1s/step - accuracy: 0.9993 - loss: 0.0038 - val_accuracy: 0.9861 - val_loss: 0.0544
Epoch 7/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 91s 1s/step - accuracy: 1.0000 - loss: 0.0019 - val_accuracy: 0.9944 - val_loss: 0.0240


In [24]:
loss, acc = model.evaluate(test_data)
print("Test Accuracy:", acc)

23/23 ━━━━━━━━━━━━━━━━━━━━ 17s 725ms/step - accuracy: 0.9917 - loss: 0.0159
Test Accuracy: 0.9916666746139526
